# Day 03: Mixture of Experts (MoE) Routing
> *Inference Engineering* — Chapter 2.2.4 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 02 (Transformer Blocks)


## Concept Overview

Mixture of Experts (MoE) replaces the dense FFN layer in a transformer with N separate expert FFNs, routing each token to only K of them (typically K=2). This allows models to scale parameter count without proportionally scaling compute — a Mixtral 8x7B has 47B parameters but uses ~13B per forward pass. The routing is learned: a gating network produces logits over experts, and top-K selection determines which experts process each token. Load balancing losses prevent router collapse (all tokens going to one expert).


In [ ]:
!pip install -q torch numpy matplotlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Top-K Gating

The router is a linear layer mapping hidden states to expert logits. Softmax + top-K selects which experts process each token, and their outputs are weighted-summed.

$$\text{gate}(x) = \text{TopK}(\text{softmax}(W_g x), K)$$


In [ ]:
class TopKRouter(nn.Module):
    def __init__(self, d_model, num_experts, top_k):
        super().__init__()
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.top_k = top_k
        self.num_experts = num_experts

    def forward(self, x):
        # x: [batch*seq, d_model]
        logits = self.gate(x)  # [B*T, num_experts]
        scores = F.softmax(logits, dim=-1)
        topk_vals, topk_idx = torch.topk(scores, self.top_k, dim=-1)
        # Normalize top-k weights
        topk_weights = topk_vals / topk_vals.sum(dim=-1, keepdim=True)
        return topk_weights, topk_idx

router = TopKRouter(d_model=256, num_experts=8, top_k=2)
x = torch.randn(32, 256)  # 32 tokens
weights, indices = router(x)
print(f'Router input shape: {x.shape}')
print(f'Top-2 weights shape: {weights.shape}')
print(f'Top-2 indices shape: {indices.shape}')
print(f'Sample token expert assignments: {indices[:4]}')
print(f'Expert load distribution: {torch.bincount(indices.flatten(), minlength=8)}')


## 2. Expert FFN and MoE Layer

Each expert is a standard FFN. The MoE layer dispatches tokens to their assigned experts, runs them in parallel, and combines the results with the gating weights.


In [ ]:
class ExpertFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class MoELayer(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k):
        super().__init__()
        self.experts = nn.ModuleList([ExpertFFN(d_model, d_ff) for _ in range(num_experts)])
        self.router = TopKRouter(d_model, num_experts, top_k)
        self.num_experts = num_experts
        self.top_k = top_k

    def forward(self, x):
        B, T, D = x.shape
        x_flat = x.view(B*T, D)
        weights, indices = self.router(x_flat)  # [B*T, k], [B*T, k]
        out = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(self.num_experts):
                mask = (indices[:, k] == e)
                if mask.any():
                    expert_out = self.experts[e](x_flat[mask])
                    out[mask] += weights[mask, k:k+1] * expert_out
        return out.view(B, T, D)

moe = MoELayer(d_model=256, d_ff=1024, num_experts=8, top_k=2)
x = torch.randn(2, 16, 256)  # batch=2, seq=16, d=256
out = moe(x)
print(f'MoE input:  {x.shape}')
print(f'MoE output: {out.shape}')
params_dense = 2 * 256 * 1024  # one FFN
params_moe = 8 * 2 * 256 * 1024  # 8 experts
print(f'Dense FFN params: {params_dense:,}')
print(f'MoE params (8 experts): {params_moe:,}')
print(f'Active params per token: {params_dense * 2:,} (top-2 experts)')
print(f'Parameter efficiency ratio: {params_moe / (params_dense * 2):.1f}x more params, same compute')


## 3. Load Balancing Loss

Without regularization, the router collapses — popular experts get stronger gradients and become more popular. The auxiliary load balancing loss penalizes uneven expert utilization:

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N \sum_{i=1}^{N} f_i \cdot P_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ and $P_i$ is the mean routing probability.


In [ ]:
def load_balance_loss(router_logits, num_experts, top_k):
    """Compute auxiliary load balancing loss (Switch Transformer style)."""
    scores = F.softmax(router_logits, dim=-1)  # [N, E]
    _, indices = torch.topk(scores, top_k, dim=-1)
    # f_i: fraction of tokens routed to expert i
    one_hot = F.one_hot(indices, num_experts).float()  # [N, k, E]
    f = one_hot.sum(dim=1).mean(dim=0)  # [E]
    # P_i: mean routing probability for expert i
    P = scores.mean(dim=0)  # [E]
    loss = num_experts * (f * P).sum()
    return loss

# Simulate balanced vs collapsed routing
N = 1000  # tokens
E = 8     # experts

# Balanced: roughly uniform
balanced_logits = torch.randn(N, E)
loss_balanced = load_balance_loss(balanced_logits, E, top_k=2)

# Collapsed: all mass on expert 0
collapsed_logits = torch.randn(N, E)
collapsed_logits[:, 0] += 10.0
loss_collapsed = load_balance_loss(collapsed_logits, E, top_k=2)

print(f'Load balance loss (balanced routing):  {loss_balanced:.4f}')
print(f'Load balance loss (collapsed routing): {loss_collapsed:.4f}')
print(f'Ratio: {loss_collapsed/loss_balanced:.1f}x higher for collapsed')

# Visualize expert load distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, logits, title in zip(axes, [balanced_logits, collapsed_logits], ['Balanced', 'Collapsed']):
    scores = F.softmax(logits, dim=-1)
    _, idx = torch.topk(scores, 2, dim=-1)
    counts = torch.bincount(idx.flatten(), minlength=E).float()
    ax.bar(range(E), counts / counts.sum())
    ax.set_xlabel('Expert ID'); ax.set_ylabel('Fraction of tokens')
    ax.set_title(f'{title} routing')
    ax.axhline(1/E, color='red', linestyle='--', label='Uniform')
    ax.legend()
plt.tight_layout()
plt.savefig('moe_routing_balance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved moe_routing_balance.png')


## 4. MoE vs Dense: Parameter Efficiency

Real-world MoE models like Mixtral 8x7B and DeepSeek MoE demonstrate that sparse activation achieves quality comparable to dense models with 2-4x fewer active FLOPs.


In [ ]:
# Compare parameter counts and active compute for real model configs
configs = {
    'Llama-3-8B (dense)': {'layers': 32, 'd_model': 4096, 'd_ff': 14336, 'num_experts': 1, 'top_k': 1},
    'Mixtral-8x7B (MoE)': {'layers': 32, 'd_model': 4096, 'd_ff': 14336, 'num_experts': 8, 'top_k': 2},
    'DeepSeek-MoE-16B':   {'layers': 28, 'd_model': 2048, 'd_ff': 1408,  'num_experts': 64, 'top_k': 6},
}

print(f'{"Model":<30} {"Total FFN Params":>18} {"Active FFN Params":>20} {"Efficiency":>12}')
print('-' * 82)
for name, cfg in configs.items():
    total = cfg['layers'] * cfg['num_experts'] * 2 * cfg['d_model'] * cfg['d_ff']
    active = cfg['layers'] * cfg['top_k'] * 2 * cfg['d_model'] * cfg['d_ff']
    eff = total / active
    print(f'{name:<30} {total/1e9:>17.2f}B {active/1e9:>19.2f}B {eff:>11.1f}x')


## Experiments: Try These

1. **Vary top-K**: Change `top_k` from 1 to 4 and measure how load distribution changes with the same router.
2. **Expert specialization**: Train a tiny MoE on synthetic data (e.g., odd vs even numbers) and inspect which expert handles which input type.
3. **Router noise**: Add Gaussian noise to router logits during training (as in Switch Transformer) and observe its effect on load balancing.


## Key Takeaways

- MoE replaces the dense FFN with N experts and routes each token to only K of them, enabling parameter scaling without proportional compute scaling.
- The router is a learned linear layer; top-K selection with weight normalization controls expert mixing.
- Without load balancing loss, routers collapse to a few popular experts — the auxiliary loss enforces uniform utilization.
- Real MoE models (Mixtral 8x7B, DeepSeek) achieve dense-model quality at 2-4x lower active FLOP counts.

## References
- *Inference Engineering* Ch 2.2.4 — Philip Kiely, Baseten Books 2026
- Shazeer et al. (2017), "Outrageously Large Neural Networks: The Sparsely-Gated MoE Layer"
- Fedus et al. (2021), "Switch Transformers"
- Jiang et al. (2024), "Mixtral of Experts"
